In [1]:
import numpy as np
import matplotlib.pyplot as plt
import s3fs

In [3]:
fs = s3fs.S3FileSystem(anon=True)

# See top-level layout
print("Top of bucket:")
for item in fs.ls("ncar-cesm2-arise/raw/"):
    print(" ", item)

# Look for ARISE-1.0 experiments — case and naming may differ
# Common naming patterns to try:
candidates = [
    "ncar-cesm2-arise/ARISE-SAI-1.0/",
    "ncar-cesm2-arise/ARISE-SAI-1.0-EXTENDED/",
]
for c in candidates:
    try:
        print(f"\nContents of {c}:")
        for item in fs.ls(c):
            print(" ", item)
    except FileNotFoundError:
        print(f"  (not found)")

Top of bucket:
  ncar-cesm2-arise/raw/
  ncar-cesm2-arise/raw/
  ncar-cesm2-arise/raw/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DELAYED-2045.001
  ncar-cesm2-arise/raw/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DELAYED-2045.002
  ncar-cesm2-arise/raw/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DELAYED-2045.003
  ncar-cesm2-arise/raw/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DELAYED-2045.004
  ncar-cesm2-arise/raw/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DELAYED-2045.005
  ncar-cesm2-arise/raw/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DELAYED-2045.006
  ncar-cesm2-arise/raw/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DELAYED-2045.007
  ncar-cesm2-arise/raw/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DELAYED-2045.008
  ncar-cesm2-arise/raw/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DELAYED-2045.009
  ncar-cesm2-arise/raw/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DELAYED-2045.010
  ncar-cesm2-arise/raw/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-LOWER-0.5.001
  ncar-cesm2-arise/raw/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-LOWER-0.5.002
  ncar-cesm2-arise/raw/b.e21.BW

In [ ]:
import s3fs
from pathlib import Path

fs = s3fs.S3FileSystem(anon=True)

# ---- CONFIGURE ----
BUCKET = "ncar-cesm2-arise"
EXPERIMENT = "raw"          # adjust after you confirm in step 1
CASE = "b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-LOWER-0.5.001"  # adjust to actual ARISE-1.0 case name
SUBDIR = "atm/proc/tseries/month_1"
LOCAL_DIR = Path("./arise_data")

VARIABLES = [
    "CLDTOT", "FLNR", "FLNS", "FLNSC", "FLNT", "FLNTC", "FLNTCLR",
    "FLUT", "FSNS", "FSNSC", "FSNT", "FSNTOA", "FSNTOAC",
    "LHFLX", "PRECT", "SHFLX", "TS",
]
# -------------------

prefix = f"{BUCKET}/{EXPERIMENT}/{CASE}/{SUBDIR}/"
print(f"Listing: {prefix}")
all_files = fs.ls(prefix)
print(f"Found {len(all_files)} files in directory\n")

# Index files by variable name (the token between '.h0.' and the date range)
def var_from_filename(path):
    name = path.split("/")[-1]
    # e.g. ....cam.h0.CLDTOT.206001-209912.nc -> CLDTOT
    parts = name.split(".")
    try:
        h0_idx = parts.index("h0")
        return parts[h0_idx + 1]
    except (ValueError, IndexError):
        return None

available = {}
for f in all_files:
    v = var_from_filename(f)
    if v:
        available.setdefault(v, []).append(f)

# Report
print("=== Verification report ===")
found, missing = [], []
for v in VARIABLES:
    if v in available:
        for f in available[v]:
            print(f"  [OK]      {v:15s} -> {f.split('/')[-1]}")
            found.append(f)
    else:
        print(f"  [MISSING] {v}")
        missing.append(v)

extras = sorted(set(available) - set(VARIABLES))
if extras:
    print(f"\nOther variables present in directory (not requested): {extras}")

print(f"\nSummary: {len(found)} files to download, {len(missing)} variables missing")

Listing: ncar-cesm2-arise/raw/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-LOWER-0.5.001/atm/proc/tseries/month_1/
Found 28770 files in directory

=== Verification report ===
  [OK]      CLDTOT          -> b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-LOWER-0.5.001.cam.h0.CLDTOT.203501-203512.nc
  [OK]      CLDTOT          -> b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-LOWER-0.5.001.cam.h0.CLDTOT.203512-203612.nc
  [OK]      CLDTOT          -> b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-LOWER-0.5.001.cam.h0.CLDTOT.203612-203712.nc
  [OK]      CLDTOT          -> b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-LOWER-0.5.001.cam.h0.CLDTOT.203712-203812.nc
  [OK]      CLDTOT          -> b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-LOWER-0.5.001.cam.h0.CLDTOT.203812-203912.nc
  [OK]      CLDTOT          -> b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-LOWER-0.5.001.cam.h0.CLDTOT.203912-204012.nc
  [OK]      CLDTOT          -> b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-LOWER-0.5.001.cam.h0.CLDTOT.204012-204112.nc
  [OK]      CLDTOT          -> b.e21.BW.f09_g17.SS

A few practical notes: fs.get is single-threaded — if you're pulling dozens of files, you can speed it up considerably with concurrent.futures.ThreadPoolExecutor (S3 parallelizes well). And these monthly time-series files are typically a few hundred MB to a couple GB each, so 18 variables × multiple members can add up fast — worth checking total size with fs.size() before kicking off a large download.

In [ ]:
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

for i, remote in enumerate(found, 1):
    local = LOCAL_DIR / remote.split("/")[-1]
    if local.exists():
        print(f"[{i}/{len(found)}] skip (exists): {local.name}")
        continue
    size_mb = fs.size(remote) / 1e6
    print(f"[{i}/{len(found)}] downloading {local.name} ({size_mb:.1f} MB)")
    fs.get(remote, str(local))

print("Done.")

In [ ]:
for member in range(1, 11): 
    # Use the case name pattern from cell 3
    case = f"b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-LOWER-0.5.{member:03d}"
    prefix = f"{BUCKET}/{EXPERIMENT}/{case}/{SUBDIR}/"
    
    print(f"\n--- Processing member {member} ({case}) ---")
    
    try:
        all_files = fs.ls(prefix)
    except FileNotFoundError:
        print(f"Directory not found for member {member}, skipping.")
        continue

    # Identify files for this member
    available = {}
    for f in all_files:
        v = var_from_filename(f)
        if v:
            available.setdefault(v, []).append(f)

    # Collect exactly the requested variables
    found = []
    for v in VARIABLES:
        if v in available:
            for f in available[v]:
                found.append(f)
    
    print(f"Found {len(found)} files to download for member {member}")
    
    # Download files
    for i, remote in enumerate(found, 1):
        local = LOCAL_DIR / remote.split("/")[-1]
        if local.exists():
            print(f"[{i}/{len(found)}] skip (exists): {local.name}")
            continue
        size_mb = fs.size(remote) / 1e6
        print(f"[{i}/{len(found)}] downloading {local.name} ({size_mb:.1f} MB)")
        fs.get(remote, str(local))